In [2]:
import pandas as pd

data_path = r"D:\NutritionAI_V1\Data\raw\health_data.csv"
df = pd.read_csv(data_path)

C:\Users\admin\AppData\Local\Temp\ipykernel_36572\3618986511.py:4: DtypeWarning: Columns (2,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072129 entries, 0 to 1072128
Data columns (total 18 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   type                  1072129 non-null  object
 1   sourceName            1072129 non-null  object
 2   sourceVersion         1072129 non-null  object
 3   unit                  1061212 non-null  object
 4   m_creationDate        1072129 non-null  object
 5   m_creationTime        1072129 non-null  object
 6   m_creationTime_am_pm  1072129 non-null  object
 7   m_creationTimeZone    1072129 non-null  int64 
 8   m_startDate           1072129 non-null  object
 9   m_startTime           1072129 non-null  object
 10  m_startTime_am_pm     1072129 non-null  object
 11  m_startTimeZone       1072129 non-null  int64 
 12  m_endDate             1072129 non-null  object
 13  m_endTime             1072129 non-null  object
 14  m_endTime_am_pm       1072129 non-null  object
 15

In [ ]:
import pandas as pd
import yaml
import os

# 1. Khai báo đường dẫn
# Sửa lại đường dẫn này trỏ tới file dữ liệu (.csv)
data_path = r"D:\NutritionAI_V1\Data\raw\health_data.csv"
schema_path = r"D:\NutritionAI_V1\src\api\data_schema\schema.yaml"

# 2. Đọc tập dữ liệu
try:
    df = pd.read_csv(data_path)
    print(f"Đã tải thành công dataset với {df.shape[0]} dòng và {df.shape[1]} cột.")
except FileNotFoundError:
    print(f"❌ Không tìm thấy file dữ liệu tại: {data_path}")
    df = pd.DataFrame() 

# 3. Khởi tạo cấu trúc Schema
schema_data = {
    "columns": {},
    "numerical_columns": [],
    "categorical_columns": [],
    "datetime_columns": [],
    "boolean_columns": []
}

# 4. Trích xuất columns và phân loại vào các nhóm
if not df.empty:
    for col_name, dtype in df.dtypes.items():
        # -- Phân loại kiểu dữ liệu --
        if pd.api.types.is_integer_dtype(dtype):
            col_type = "integer"
            schema_data["numerical_columns"].append(col_name)
            
        elif pd.api.types.is_float_dtype(dtype):
            col_type = "float"
            schema_data["numerical_columns"].append(col_name)
            
        elif pd.api.types.is_bool_dtype(dtype):
            col_type = "boolean"
            schema_data["boolean_columns"].append(col_name)
            
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            col_type = "datetime"
            schema_data["datetime_columns"].append(col_name)
            
        else:
            col_type = "categorical" # Xử lý Object/String
            schema_data["categorical_columns"].append(col_name)

        # -- Ghi chi tiết từng cột --
        schema_data["columns"][col_name] = {
            "type": col_type,
            "nullable": df[col_name].isnull().any().item(), # .item() để chuyển từ numpy bool sang python bool chuẩn
            "description": "" 
        }

# 5. Loại bỏ các list rỗng (nếu dataset không có kiểu dữ liệu đó)
# Ví dụ: Nếu không có cột boolean, nó sẽ không in ra "boolean_columns: []"
schema_data = {k: v for k, v in schema_data.items() if v}

# 6. Đọc nội dung schema cũ (nếu có) để giữ lại các config tuỳ chỉnh khác (như target_column)
if os.path.exists(schema_path):
    with open(schema_path, "r", encoding="utf-8") as file:
        existing_schema = yaml.safe_load(file) or {}
        # Hợp nhất: cập nhật các key mới tạo vào schema cũ
        existing_schema.update(schema_data)
        schema_data = existing_schema
else:
    os.makedirs(os.path.dirname(schema_path), exist_ok=True)

# 7. Ghi ra file YAML
with open(schema_path, "w", encoding="utf-8") as file:
    yaml.dump(schema_data, file, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo schema và phân nhóm thành công. Lưu tại: {schema_path}")

C:\Users\admin\AppData\Local\Temp\ipykernel_36572\3847332735.py:12: DtypeWarning: Columns (2,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


Đã tải thành công dataset với 1072129 dòng và 18 cột.
✅ Đã tạo schema và phân nhóm thành công. Lưu tại: D:\NutritionAI_V1\src\api\data_schema\schema.yaml
